# Data Cleaning

## Overview
Prepare clean, leak-free data for modeling by removing cancelled/diverted flights, 
dropping post-departure features, and creating chronological train/validation/test splits.


In [1]:
from pyprojroot import here
print("Project root:", here())

Project root: C:\Users\hanis\Flight\flight-delay-prediction


## Step 1: Filter Cancelled & Diverted Flights

Cancelled and diverted flights have no valid `DepDelay` value, so they cannot be 
used for training or evaluation. Removing them leaves 2,947,844 flights (35,285 removed).

In [2]:
import pandas as pd

df = pd.read_parquet(here("data/processed/flights.parquet"))
print(f"Before filtering: {len(df)} rows")

# Remove cancelled and diverted flights (no valid DepDelay)
df_clean = df[(df['Cancelled'] == 0) & (df['Diverted'] == 0)].copy()
print(f"After filtering: {len(df_clean)} rows")
print(f"Rows removed: {len(df) - len(df_clean)}")

Before filtering: 2983129 rows
After filtering: 2947844 rows
Rows removed: 35285


## Step 2: Drop Leaky Columns

Remove all features unavailable 2 hours before scheduled departure:
- Actual times: `DepTime`, `ArrTime`, `WheelsOff`, `WheelsOn`, `TaxiIn`, `TaxiOut`
- Delay information: `ArrDelay`, `ActualElapsedTime`, `AirTime`
- Delay attribution: `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay`
- Diversion details: all `Div*` columns
- Cancellation codes: `Cancelled`, `CancellationCode`, `Diverted`

Dropped 73 columns → 41 remaining (schedule, airline, origin, destination, target).


In [3]:
# Columns that are NOT available 2 hours before scheduled departure
leaky_cols = [
    'DepTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15',
    'ActualElapsedTime', 'AirTime', 'TaxiOut', 'TaxiIn',
    'WheelsOff', 'WheelsOn', 'CRSElapsedTime',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
    'FirstDepTime', 'TotalAddGTime', 'LongestAddGTime',
    'DivReachedDest', 'DivActualElapsedTime', 'DivArrDelay', 'DivDistance',
    'Cancelled', 'CancellationCode', 'Diverted'
]

# Also drop all Div* columns (diversion details)
div_cols = [col for col in df_clean.columns if col.startswith('Div')]

cols_to_drop = leaky_cols + div_cols

print(f"Columns to drop: {len(cols_to_drop)}")
print(f"Before drop: {df_clean.shape[1]} columns")

df_clean = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns])

print(f"After drop: {df_clean.shape[1]} columns")
print(f"\nRemaining columns:")
print(df_clean.columns.tolist())

Columns to drop: 73
Before drop: 109 columns
After drop: 41 columns

Remaining columns:
['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'CRSArrTime', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Flights', 'Distance', 'DistanceGroup']


# Drop missing value columns
We need to drop the columns with 100% missing values. But most of the missing values come from div_cols which we already dropped.
Let us cross check.

In [4]:
missing_pct = (df_clean.isnull().sum() / len(df)) * 100
print(missing_pct[missing_pct > 95].sort_values(ascending=False))

Series([], dtype: float64)


As we can see all the 100% missing columns are all div_columns(Since result is empty)

## Step 3: Dropping Obviously Invalid Values

After EDA, we realised that there are no such values in our data. Hence we can skip this step

## Step 4: Temporal Split (Chronological, No Leakage)

Split by month to prevent future information leaking into training:
- **Train:** August–October 2024 (1.79M flights, 17.2% delayed)
- **Validation:** November 2024 (572K flights, 14.4% delayed)
- **Test:** December 2024 (585K flights, 21.6% delayed)

Target rates vary by month (realistic—December weather impacts delays). 
Model will train on 17% baseline and generalize to 21% test baseline.


In [5]:
# Temporal split: train on past, validate/test on future
# Your data: Aug-Dec 2024

df_clean.to_parquet(here("data/processed/clean.parquet"), index=False) # save the df_clean for relationship EDA

train = df_clean[df_clean['Month'].isin([8, 9, 10])].copy()
val = df_clean[df_clean['Month'] == 11].copy()
test = df_clean[df_clean['Month'] == 12].copy()

print(f"Train (Aug-Oct): {len(train)} rows")
print(f"Val  (Nov):      {len(val)} rows")
print(f"Test (Dec):      {len(test)} rows")
print(f"Total:           {len(train) + len(val) + len(test)} rows")

print(f"\nTarget balance in each split:")
print(f"Train delayed%: {(train['DepDel15'].sum() / len(train) * 100):.2f}%")
print(f"Val delayed%:   {(val['DepDel15'].sum() / len(val) * 100):.2f}%")
print(f"Test delayed%:  {(test['DepDel15'].sum() / len(test) * 100):.2f}%")

Train (Aug-Oct): 1791079 rows
Val  (Nov):      571933 rows
Test (Dec):      584832 rows
Total:           2947844 rows

Target balance in each split:
Train delayed%: 17.23%
Val delayed%:   14.38%
Test delayed%:  21.59%


## Result
3 clean, leak-free parquet files ready for feature engineering.


In [6]:
# Save splits for feature engineering phase
train.to_parquet(here("data/processed/train.parquet"))
val.to_parquet(here("data/processed/val.parquet"))
test.to_parquet(here("data/processed/test.parquet"))

print("Splits saved:")
print(f"  data/processed/train.parquet")
print(f"  data/processed/val.parquet")
print(f"  data/processed/test.parquet")

Splits saved:
  data/processed/train.parquet
  data/processed/val.parquet
  data/processed/test.parquet
